# How to Talk So AI Will Learn: A Flippy Implementation

This notebook re-implements the computational models from **Sumers et al. (2022)** — *"How to talk so AI will learn: Instructions, descriptions, and autonomy"* (NeurIPS 2022) — using **flippy**, a probabilistic programming language embedded in Python.

The paper formalizes how humans communicate reward information to AI agents using two types of language:
- **Instructions**: Tell the listener which action to take (e.g., "Take the spotted red mushroom")
- **Descriptions**: Provide information about the reward function (e.g., "Spotted mushrooms are worth +1")

The models are built on the **Rational Speech Acts (RSA)** framework, which uses Bayesian inference to model pragmatic communication. This is a natural fit for flippy's probabilistic programming primitives.

## 1. Setup and Installation

Install flippy and import all necessary packages.

In [ ]:
# Install flippy if not already installed
# flippy is a probabilistic programming language from the CoDec Lab
!pip install flippy-lang
!pip install matplotlib numpy pandas

In [ ]:
import itertools
import math
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# flippy provides probabilistic programming primitives:
# - flip(p): sample a Bernoulli random variable
# - draw_from(items): sample uniformly from a collection
# - condition(c): condition on an observation (weight the current trace)
# - factor(score): add a log-probability score to the current trace
# - @infer: decorator that runs inference over a stochastic function
from flippy import infer, flip, draw_from, condition, factor

## 2. Domain Configuration: The Mushroom Foraging World

We define the contextual bandit setting from the paper. Mushrooms have three features — **size** **color** and **texture** — each contributing independently to the mushroom's reward value. A **state** (mushroom patch) is a set of 3 mushrooms from which the agent must choose one.

In [ ]:
# --- Feature definitions ---
# Each mushroom has a size, a color, and a texture
SIZES = ["small", "medium", "large"]
COLORS = ["red", "green", "blue"]
TEXTURES = ["spotted", "solid", "striped"]
FEATURES = SIZES + COLORS + TEXTURES  # all 9 features

# --- Actions ---
# An action = picking a specific mushroom (one of 3x3x3 = 27 possible types)
ACTIONS = [
    {"size": s, "color": c, "texture": t}
    for s, c, t in itertools.product(SIZES, COLORS, TEXTURES)
]
print(f"Total possible mushroom types: {len(ACTIONS)}")
print("Examples:", ACTIONS[:3])

In [ ]:
# --- States ---
# A state (patch) = a set of 3 mushrooms drawn from the 9 types
# C(27,3) = 2925 possible states
ALL_STATES = [list(combo) for combo in itertools.combinations(ACTIONS, 3)]
print(f"Total possible states (patches): {len(ALL_STATES)}")
print("Example state:", ALL_STATES[0])

In [ ]:
# --- True reward function ---
# R(mushroom) = w_color + w_texture  (Eq. 4 in the paper)
# These are the ground-truth feature weights the speaker knows
TRUE_REWARDS = {
    "small": 1,    # small mushrooms are very good
    "medium": 0,      # medium mushrooms are neutral
    "large": -1,    # large mushrooms are very bad
    "green": 1,    # green mushrooms are very good
    "red": 0,      # red mushrooms are neutral
    "blue": -1,    # blue mushrooms are very bad
    "spotted": 1,  # spotted texture adds value
    "solid": 0,    # solid texture is neutral
    "striped": -1  # striped texture subtracts value
}


def compute_reward(action, weights):
    """Compute reward for a mushroom given feature weights (Eq. 4).
    R(a, w) = w^T * phi(a) = w_size + w_color + w_texture
    """
    return weights[action["size"]] + weights[action["color"]] + weights[action["texture"]]


# Show all mushroom values under the true reward function
print("Mushroom values under true rewards:")
for a in ACTIONS:
    val = compute_reward(a, TRUE_REWARDS)
    print(f"  {a['size']:>4} {a['color']:>5} {a['texture']:>7} -> {val:+d}")

## 3. Defining the Utterance Space

The speaker only produces **instructional** utterances. An instruction names a specific mushroom by its three features: *"Take the small red spotted mushroom"* (3 × 3 × 3 = 27 possible instructions, one per mushroom type).

In [ ]:
# --- Instructions ---
# Each instruction tells the listener to take a specific mushroom.
# Stress is encoded as a per-feature dict where 1 = stressed, 0 = normal.
# E.g. stress={"size": 1, "color": 0, "texture": 0} means the size word is stressed,
# as in "take the LARGE red spotted mushroom".

# --- Baseline instructions (no stress): 27 utterances, one per mushroom type ---
NO_STRESS = {"size": 0, "color": 0, "texture": 0}

INSTRUCTIONS = [
    {"type": "instruction",
     "size": a["size"], "color": a["color"], "texture": a["texture"],
     "stress": dict(NO_STRESS)}
    for a in ACTIONS
]

# --- Stress-augmented utterance space ---
# 2^3 = 8 possible stress patterns per mushroom => 27 * 8 = 216 utterances.
STRESS_PATTERNS = [
    {"size": s, "color": c, "texture": t}
    for s, c, t in itertools.product([0, 1], repeat=3)
]

STRESSED_INSTRUCTIONS = [
    {"type": "instruction",
     "size": a["size"], "color": a["color"], "texture": a["texture"],
     "stress": dict(stress)}
    for a in ACTIONS
    for stress in STRESS_PATTERNS
]

# --- Descriptions: dropped for this project (stress extension uses instructions only) ---
# Each description states a feature-value pair: "[feature] is worth [value]"
# 9 features x 5 possible values (-2 to +2) = 45 descriptions
# POSSIBLE_VALUES = list(range(-2, 3))  # original: 5 values [-2, -1, 0, 1, 2]
# Capped to {-1, 0, 1} for tractability now that we have 9 features.
# 5^9 = 1.95M worlds is intractable; 3^9 = 19,683 is comparable to the original
# paper's 5^6 = 15,625.
POSSIBLE_VALUES = list(range(-1, 2))  # [-1, 0, 1]
# DESCRIPTIONS = [
#     {"type": "description", "feature": f, "value": v}
#     for f, v in itertools.product(FEATURES, POSSIBLE_VALUES)
# ]

# --- Experimental subset (also dropped) ---
# EXP_FEATURES = ["green", "spotted", "striped", "blue"]
# EXP_VALUES = [-2, -1, 1, 2]
# EXP_DESCRIPTIONS = [
#     {"type": "description", "feature": f, "value": v}
#     for f, v in itertools.product(EXP_FEATURES, EXP_VALUES)
# ]

# Default utterance set = baseline (no stress). The stress-augmented arm of the
# comparison can be turned on by passing STRESSED_INSTRUCTIONS instead.
ALL_UTTERANCES = INSTRUCTIONS

print(f"Baseline instructions (no stress):     {len(INSTRUCTIONS)}")
print(f"Stress patterns per mushroom:          {len(STRESS_PATTERNS)}")
print(f"Stress-augmented instructions:         {len(STRESSED_INSTRUCTIONS)}")
print(f"Total utterances (default = baseline): {len(ALL_UTTERANCES)}")
print(f"Possible reward values per feature:    {POSSIBLE_VALUES}")

## 4. Possible Reward Worlds

The pragmatic listener must reason over all possible reward functions. Each reward function assigns a value in {-1, 0, +1} to each of the 9 features, giving 3⁹ = **19,683 possible worlds**.

*(The original paper used 5⁶ = 15,625 worlds with reward values in {-2…+2} over 6 features. Adding `size` would push us to 5⁹ ≈ 1.95M worlds; we cap reward values to {-1, 0, +1} so the world space stays comparable to the original.)*

In [ ]:
def generate_all_reward_worlds(features, possible_values):
    """Generate all possible reward functions (feature-value assignments).
    Each world is a dict mapping every feature to a value.
    """
    worlds = [{}]  # start with one empty world
    for f in features:
        # for each feature, expand every existing world with every possible value
        worlds = [
            {**w, f: v}
            for w in worlds
            for v in possible_values
        ]
    return worlds


# All possible reward worlds (uniform prior)
ALL_REWARDS = generate_all_reward_worlds(FEATURES, POSSIBLE_VALUES)
print(f"Total possible reward worlds: {len(ALL_REWARDS)}")
print(f"Example world: {ALL_REWARDS[0]}")

## 5. Utility Functions

Helper functions used throughout the models: softmax for converting utilities to probabilities, and formatting utilities.

In [ ]:
def softmax(values, temperature=1.0):
    """Compute softmax probabilities from a list of values.
    Higher temperature -> more uniform; lower -> more peaked.
    """
    # scale by temperature
    scaled = [v * temperature for v in values]
    # subtract max for numerical stability
    max_val = max(scaled)
    exps = [math.exp(s - max_val) for s in scaled]
    total = sum(exps)
    return [e / total for e in exps]


def utt_to_string(utt):
    """Convert an utterance dict to a readable string.

    Stressed feature words are rendered in CAPS. Defaults to no stress
    when the field is absent (so older demo cells without a stress field
    keep working).
    """
    if utt["type"] == "instruction":
        stress = utt.get("stress", {"size": 0, "color": 0, "texture": 0})
        words = []
        for f in ("size", "color", "texture"):
            w = utt[f]
            words.append(w.upper() if stress.get(f, 0) == 1 else w)
        return f'Instruction: Take {" ".join(words)}'
    # Descriptions dropped:
    # else:
    #     return f'Description: {utt["feature"]} is {utt["value"]:+d}'


# Quick test
print(softmax([1, 2, 3], temperature=1.0))
print(utt_to_string(INSTRUCTIONS[0]))                                  # no stress
print(utt_to_string(STRESSED_INSTRUCTIONS[0]))                         # all-zero stress pattern
# Pick an instruction with stress on size only ("LARGE red spotted")
example_stressed = next(
    u for u in STRESSED_INSTRUCTIONS
    if u["size"] == "large" and u["color"] == "red" and u["texture"] == "spotted"
    and u["stress"] == {"size": 1, "color": 0, "texture": 0}
)
print(utt_to_string(example_stressed))
# print(utt_to_string(DESCRIPTIONS[0]))  # descriptions dropped

## 6. The Literal Listener (L0)

The **literal listener** interprets utterances at face value, without reasoning about the speaker's intentions.

Given an **instruction**, L0 picks the instructed mushroom if available, otherwise chooses randomly (Eq. 8). The description path is dropped for this project.

In [ ]:
class LiteralListener:
    """L0: interprets instructions literally with a stress-aware graded match.

    score(a, u) = sum over features f of  weight_f * 1[a[f] == u[f]]
    where weight_f = w_stress if u.stress[f] == 1 else w_normal.
    P_L0(a | u, s) ~ softmax(alpha_L * score).

    With w_stress == w_normal, every feature is weighted equally and (for high
    alpha_L) the exact instructed mushroom is picked near-deterministically when
    present -- recovering the original RSA semantics. Setting w_stress > w_normal
    makes stressed features criterial: when no exact match exists in the patch,
    L0 prefers actions that match on stressed features, with unstressed features
    serving as tiebreakers.
    """

    def __init__(self, alpha_L=3, w_stress=5.0, w_normal=1.0, all_states=None):
        # alpha_L controls how greedily the listener picks actions (softmax temperature)
        self.alpha_L = alpha_L
        # per-feature weights for the match score
        self.w_stress = w_stress
        self.w_normal = w_normal
        # all possible states, used for computing future (generalization) rewards
        self.all_states = all_states or ALL_STATES
        # cache for expensive feature count computations
        self._cache = {}

    def prob_action_given_instruction(self, action, context, instruction):
        """P(a | u_instruction, s): stress-weighted graded match (replaces Eq. 8).

        Stress dict on the instruction tells us which feature words were stressed;
        stressed features get weight w_stress, unstressed get w_normal.
        """
        # --- Original deterministic-or-uniform semantics (replaced by graded match below) ---
        # target = {
        #     "size": instruction["size"],
        #     "color": instruction["color"],
        #     "texture": instruction["texture"],
        # }
        # matches = [a for a in context if a == target]
        # if not matches:
        #     return 1.0 / len(context)
        # if action == matches[0]:
        #     return 1.0
        # return 0.0

        stress = instruction.get("stress", {"size": 0, "color": 0, "texture": 0})

        def match_score(a):
            s = 0.0
            for f in ("size", "color", "texture"):
                weight = self.w_stress if stress.get(f, 0) == 1 else self.w_normal
                if a[f] == instruction[f]:
                    s += weight
            return s

        scores = [match_score(a) for a in context]
        this_score = match_score(action)

        # softmax over match scores (subtract max for numerical stability)
        max_s = max(scores)
        exps = [math.exp(self.alpha_L * (s - max_s)) for s in scores]
        total = sum(exps)
        return math.exp(self.alpha_L * (this_score - max_s)) / total

    # --- Description path (dropped for this project) ---
    # def prob_action_given_description(self, action, context, description):
    #     """P(a | u_description, s): probability of choosing action given description (Eq. 10)."""
    #     beliefs = {f: 0 for f in FEATURES}
    #     beliefs[description["feature"]] = description["value"]
    #     return self._prob_action_from_beliefs(action, context, beliefs)
    #
    # def _prob_action_from_beliefs(self, action, context, beliefs):
    #     """Softmax action selection given beliefs about feature values."""
    #     logits = [compute_reward(a, beliefs) * self.alpha_L for a in context]
    #     this_logit = compute_reward(action, beliefs) * self.alpha_L
    #     total = sum(math.exp(l) for l in logits)
    #     return math.exp(this_logit) / total

    def present_feature_counts(self, utt, context):
        """Expected feature counts when listener acts on utterance in this context.

        Sums P(a|u,s) * features(a) over all actions a in context.
        """
        key = str(context) + str(utt)
        if key not in self._cache:
            counts = defaultdict(float)

            # only instructions are supported now
            probs = [self.prob_action_given_instruction(a, context, utt) for a in context]
            # else:
            #     probs = [self.prob_action_given_description(a, context, utt) for a in context]

            # accumulate expected feature counts (weighted by action probability)
            for a, p in zip(context, probs):
                counts[a["size"]] += p
                counts[a["color"]] += p
                counts[a["texture"]] += p

            self._cache[key] = dict(counts)
        return self._cache[key]

    def future_feature_counts(self, utt):
        """Expected feature counts averaged over ALL possible future states."""
        key = "future_" + str(utt)
        if key not in self._cache:
            all_counts = [Counter(self.present_feature_counts(utt, s)) for s in self.all_states]
            total = Counter()
            for c in all_counts:
                total += c
            avg = {k: v / len(self.all_states) for k, v in total.items()}
            self._cache[key] = avg
        return self._cache[key]

    def present_rewards(self, utt, context, reward_weights):
        """Expected reward from following utterance in this specific context (Eq. 5)."""
        counts = self.present_feature_counts(utt, context)
        return sum(counts.get(f, 0) * reward_weights[f] for f in reward_weights)

    def future_rewards(self, utt, context, reward_weights):
        """Expected reward from following utterance across all future states (Eq. 6)."""
        counts = self.future_feature_counts(utt)
        return sum(counts.get(f, 0) * reward_weights[f] for f in reward_weights)


# Create a literal listener with default parameters
L0 = LiteralListener(alpha_L=3, w_stress=5.0, w_normal=1.0)
print("Literal Listener (L0) initialized with alpha_L=3, w_stress=5.0, w_normal=1.0")

### 6.1 Test the Literal Listener

Verify L0 behaves correctly: given an instruction to take a specific mushroom, L0 should pick it deterministically when present.

In [ ]:
# Example state: a patch of 3 mushrooms (3-feature: size x color x texture)
example_state = [
    {"size": "small",  "color": "red",   "texture": "spotted"},   # value = 1 + 0 + 1 = +2
    {"size": "medium", "color": "green", "texture": "solid"},     # value = 0 + 1 + 0 = +1
    {"size": "large",  "color": "blue",  "texture": "striped"},   # value = -1 + -1 + -1 = -3
]

# Test instruction: "Take the small red spotted mushroom"
instr = {"type": "instruction", "size": "small", "color": "red", "texture": "spotted"}
print("=== Instruction: Take small red spotted ===")
for a in example_state:
    p = L0.prob_action_given_instruction(a, example_state, instr)
    print(f"  P(take {a['size']} {a['color']} {a['texture']}) = {p:.2f}")

# --- Description test (dropped for this project) ---
# desc = {"type": "description", "feature": "green", "value": 2}
# print(f"\n=== Description: {utt_to_string(desc)} ===")
# for a in example_state:
#     p = L0.prob_action_given_description(a, example_state, desc)
#     print(f"  P(take {a['color']} {a['texture']}) = {p:.3f}")

## 7. The Speaker Model (S1) with Flippy

The **speaker** chooses utterances to maximize the listener's expected reward, balancing **present** utility (what helps now) and **future** utility (what generalizes).

The speaker's objective (Eq. 7) weighs present vs. future rewards by the **horizon** H:

$$U_{S_1}(u \mid w, s, H) = \frac{1}{H} U_{\text{Present}} + (1 - \frac{1}{H}) U_{\text{Future}}$$

We implement this as a flippy probabilistic program using `factor()` to weight utterances by their utility.

In [ ]:
def make_speaker(listener, alpha_S=10, reward_weights=None, utterances=None):
    """Create a flippy speaker. Returns (get_speaker, speaker_utility).

    `get_speaker(context, horizon)` precomputes utterance utilities OUTSIDE the
    @infer'd region (because flippy's enumeration cache requires all user-function
    args to be hashable, but `context` is a list of dicts). It returns a no-arg
    closure suitable for use inside @infer:

        speak = get_speaker(demo_state, horizon=1)
        @infer
        def demo():
            return speak()           # only flippy primitives inside @infer
    """
    if reward_weights is None:
        reward_weights = TRUE_REWARDS
    if utterances is None:
        utterances = ALL_UTTERANCES

    def speaker_utility(utt, context, horizon):
        """Eq. 7: blends present (this state) and future (generalization) reward."""
        present = listener.present_rewards(utt, context, reward_weights)
        future = listener.future_rewards(utt, context, reward_weights)
        return (present + (horizon - 1) * future) / horizon

    def get_speaker(context, horizon=1):
        # Pre-compute utility for every utterance (deterministic; runs outside @infer)
        utilities = [speaker_utility(u, context, horizon) for u in utterances]

        def speak():
            """Sample an utterance index weighted by alpha_S * utility."""
            idx = draw_from(len(utterances))
            factor(alpha_S * utilities[idx])
            return idx

        return speak

    return get_speaker, speaker_utility


print("Speaker model factory defined.")

### 7.1 Speaker Utterance Probabilities (Non-Flippy Baseline)

For comparison and efficiency, we also implement the speaker as a direct softmax computation (matching the original paper's implementation). This is useful for the pragmatic listener, which needs to evaluate speaker probabilities thousands of times.

In [ ]:
class Speaker:
    """S1: chooses utterances to maximize the literal listener's expected reward.

    Uses softmax over utterance utilities (Eq. 7).
    """

    def __init__(self, listener, alpha_S=10, reward_weights=None, utterances=None):
        self.listener = listener
        self.alpha_S = alpha_S  # speaker rationality parameter
        self.reward_weights = reward_weights or TRUE_REWARDS
        self.utterances = utterances or ALL_UTTERANCES

    def utterance_utilities(self, context, horizon=1, reward_weights=None):
        """Compute utility of every utterance in the given context (Eq. 7)."""
        w = reward_weights or self.reward_weights
        utilities = []
        for utt in self.utterances:
            # present reward: how well does the listener do in THIS state?
            present = self.listener.present_rewards(utt, context, w)
            # future reward: how well does the listener do in UNKNOWN future states?
            future = self.listener.future_rewards(utt, context, w)
            # horizon-weighted combination (Eq. 7)
            utility = (present + (horizon - 1) * future) / horizon
            utilities.append(utility)
        return utilities

    def utterance_probabilities(self, context, horizon=1, reward_weights=None):
        """Softmax distribution over utterances (speaker's production distribution)."""
        utilities = self.utterance_utilities(context, horizon, reward_weights)
        return softmax(utilities, temperature=self.alpha_S)

    def single_utterance_probability(self, utt, context, horizon=1, reward_weights=None):
        """Probability of a specific utterance."""
        probs = self.utterance_probabilities(context, horizon, reward_weights)
        idx = self.utterances.index(utt)
        return probs[idx]


# Create the speaker with default literal listener
S1 = Speaker(L0, alpha_S=10)
print("Speaker (S1) initialized with alpha_S=10")

### 7.2 Flippy Speaker Demo

Let's use flippy's `@infer` decorator to compute the speaker's utterance distribution for a specific context and horizon. This demonstrates how flippy naturally represents the softmax computation as probabilistic inference.

In [ ]:
# Create a small utterance set for demo (3-feature mushrooms)
demo_state = [
    {"size": "small",  "color": "red",   "texture": "spotted"},   # value +2
    {"size": "medium", "color": "green", "texture": "solid"},     # value +1
    {"size": "large",  "color": "blue",  "texture": "striped"},   # value -3
]

# Use the 27 instructions for this demo
demo_utterances = INSTRUCTIONS

# Build the flippy speaker factory.
get_speak, speak_utility = make_speaker(
    L0, alpha_S=10,
    reward_weights=TRUE_REWARDS,
    utterances=demo_utterances
)

# Pre-compute utilities OUTSIDE @infer (flippy's enumeration cache can't hash
# list-of-dict args, so the @infer'd function must only call flippy primitives).
speak_for_demo = get_speak(demo_state, horizon=1)


@infer
def flippy_speaker_demo():
    return speak_for_demo()


# Get the result distribution
result = flippy_speaker_demo()
result_dict = dict(result)

# Display the top utterances
print("Flippy speaker distribution (H=1, instructions only):")
for idx, prob in sorted(result_dict.items(), key=lambda x: -x[1]):
    if prob > 0.001:  # only show non-negligible probabilities
        utt = demo_utterances[idx]
        print(f"  {utt_to_string(utt):>50}  P = {prob:.4f}")

## 8. Speaker Behavior Across Horizons

*This section originally compared instructions vs descriptions across horizons (Fig. 2 of the paper). Since this project uses instructions only, the comparison code below is commented out — to be repurposed later for a stress vs no-stress comparison.*

In [ ]:
# --- Original instructions-vs-descriptions horizon comparison (commented out: descriptions dropped) ---
# horizons = list(range(1, 11))  # H = 1 to 10
#
# L0_fresh = LiteralListener(alpha_L=3)
# speaker_instr = Speaker(L0_fresh, alpha_S=10, utterances=INSTRUCTIONS)
# speaker_desc  = Speaker(L0_fresh, alpha_S=10, utterances=DESCRIPTIONS)
#
#
# def avg_speaker_reward(speaker, horizon, n_states=None):
#     """Average speaker reward across all (or sampled) states for a given horizon."""
#     states = ALL_STATES if n_states is None else ALL_STATES[:n_states]
#     total = 0.0
#     for state in states:
#         probs = speaker.utterance_probabilities(state, horizon=horizon)
#         utils = speaker.utterance_utilities(state, horizon=horizon)
#         expected = sum(p * u for p, u in zip(probs, utils))
#         total += expected
#     return total / len(states)
#
#
# print("Computing speaker rewards across horizons (using 20 sample states)...")
# instr_rewards = [avg_speaker_reward(speaker_instr, h, n_states=20) for h in horizons]
# desc_rewards  = [avg_speaker_reward(speaker_desc, h, n_states=20) for h in horizons]
# print("Done!")

In [ ]:
# --- Original instruction-vs-description plot (commented out) ---
# plt.figure(figsize=(8, 5))
# plt.plot(horizons, instr_rewards, 'k--', label='Instructions', linewidth=2)
# plt.plot(horizons, desc_rewards, 'gray', label='Descriptions', linewidth=2)
# plt.xlabel('Speaker Horizon H', fontsize=13)
# plt.ylabel('Speaker Rewards', fontsize=13)
# plt.title('Speaker Rewards by Horizon (cf. Fig. 2B)', fontsize=14)
# plt.legend(fontsize=12)
# plt.xticks(horizons)
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

## 9. The Pragmatic Listener (L1) with Flippy

The **pragmatic listener** inverts the speaker model using Bayesian inference (Eq. 3, Eq. 11):

$$L_1(w \mid s, u, H) \propto S_1(u \mid w, s, H) \cdot P(w)$$

Given an utterance, L1 reasons: *"If a rational speaker chose this utterance, what must the reward function be?"*

This is where flippy truly shines — we can express this as a natural probabilistic program using `draw_from` for the prior over worlds and `factor` for the speaker likelihood.

In [ ]:
def make_pragmatic_listener_flippy(speaker, possible_rewards, utterances=None):
    """Create a pragmatic listener as a flippy probabilistic program.

    Returns `get_listener(utt, context, horizon)` which precomputes
    log-likelihoods OUTSIDE the @infer'd region (flippy's enumeration cache
    can't hash list-of-dict args, so the @infer'd function must only call
    flippy primitives) and returns a no-arg closure to use inside @infer:

        listen = get_listener(observed_utt, context, horizon=1)
        @infer
        def l1():
            return listen()

    Implements Eq. 11:
        L1(w | s, u, H) ~ S1(u | w, s, H) * P(w)
    """
    if utterances is None:
        utterances = ALL_UTTERANCES

    def get_listener(utt, context, horizon=1):
        # Pre-compute speaker log-probabilities for each candidate world (deterministic)
        log_likelihoods = [
            math.log(speaker.single_utterance_probability(
                utt, context, horizon=horizon, reward_weights=w
            ) + 1e-300)
            for w in possible_rewards
        ]

        def listen():
            # Sample a world index from the uniform prior, weight by speaker likelihood
            world_idx = draw_from(len(possible_rewards))
            factor(log_likelihoods[world_idx])
            return world_idx

        return listen

    return get_listener


print("Pragmatic listener (flippy) factory defined.")

### 9.1 Pragmatic Listener Demo (Small World)

Running exact inference over all 15,625 reward worlds is expensive. For demonstration, we use a **reduced world** where only one feature varies (e.g., only the "spotted" value is unknown). This shows the core mechanism clearly.

In [ ]:
# Small world subspace: vary one feature per dimension (size, color, texture).
# All other features are pinned to TRUE_REWARDS.
# 3 dimensions x 3 values = 27 small worlds.
#
# The varying features (large, red, spotted) are chosen so the demo can showcase
# stress on any of the three feature types — e.g. "take LARGE red spotted" stresses
# size, "take large RED spotted" stresses color, etc.
VARYING = ("large", "red", "spotted")

small_worlds = []
for large_val, red_val, spotted_val in itertools.product(POSSIBLE_VALUES, repeat=3):
    world = dict(TRUE_REWARDS)  # copy true rewards
    world["large"]   = large_val
    world["red"]     = red_val
    world["spotted"] = spotted_val
    small_worlds.append(world)

print(f"Small world set ({len(small_worlds)} worlds varying {VARYING}):")
for i, w in enumerate(small_worlds[:6]):
    print(f"  World {i}: large={w['large']:+d}, red={w['red']:+d}, spotted={w['spotted']:+d}")
print("  ...")

# Create speaker and listener for the small world.
# Speaker uses the FULL stress-augmented utterance set (216 instructions) so that
# both no-stress and stressed observations are valid speaker outputs — required
# for the stress comparison demo below.
L0_small = LiteralListener(alpha_L=3, w_stress=5.0, w_normal=1.0)
S1_small = Speaker(L0_small, alpha_S=10, utterances=STRESSED_INSTRUCTIONS)

# Build the flippy pragmatic listener factory
get_listener = make_pragmatic_listener_flippy(S1_small, small_worlds)

# Observation: speaker says "Take the small red spotted mushroom" (no stress)
observed_utt = {"type": "instruction",
                "size": "small", "color": "red", "texture": "spotted",
                "stress": {"size": 0, "color": 0, "texture": 0}}
# Original (description) observation, dropped:
# observed_utt = {"type": "description", "feature": "spotted", "value": 1}
test_context = demo_state

print(f"\nObserved utterance: {utt_to_string(observed_utt)}")
print(f"Context: {test_context}")

# Pre-compute speaker likelihoods OUTSIDE @infer (closure captures them)
listen_for_demo = get_listener(observed_utt, test_context, horizon=1)


@infer
def l1_inference():
    return listen_for_demo()


posterior = dict(l1_inference())

# Marginalize the joint posterior to per-feature distributions
def marginal(posterior, worlds, feature):
    """P(world[feature] = v | utterance), summed over the other varying features."""
    out = {v: 0.0 for v in POSSIBLE_VALUES}
    for idx, prob in posterior.items():
        out[worlds[idx][feature]] += prob
    return out


print(f"\nPragmatic listener marginals given '{utt_to_string(observed_utt)}':")
for f in VARYING:
    m = marginal(posterior, small_worlds, f)
    print(f"  {f}:")
    for v in POSSIBLE_VALUES:
        bar = '#' * int(m[v] * 50)
        print(f"    {f} = {v:+d}:  {m[v]:.4f}  {bar}")

### 9.2 Pragmatic Listener: Direct Computation (Full Scale)

For the full model (all 15,625 worlds), we compute the posterior directly using Bayes' rule without flippy enumeration, matching the original paper's approach. This is equivalent to what flippy computes, but avoids enumerating 15K+ states.

In [ ]:
class PragmaticListener:
    """L1: inverts the speaker model via Bayesian inference (Eq. 3, 11).

    Given an utterance, infers the most likely reward function
    by computing P(w|u) ~ P(u|w) * P(w) over all possible worlds.
    """

    def __init__(self, speaker, alpha_L=3, possible_rewards=None):
        self.speaker = speaker
        self.alpha_L = alpha_L
        self.possible_rewards = possible_rewards or ALL_REWARDS
        self.all_states = ALL_STATES

    def inference(self, utt, context, horizon):
        """Compute posterior over reward worlds given utterance (Eq. 11).

        If horizon is a list, marginalizes over horizons (Eq. 12 - latent horizon).
        Returns a DataFrame with columns for each feature + likelihoods + probability.
        """
        if isinstance(horizon, list):
            # latent horizon model: marginalize over multiple horizons
            return self._multihorizon_inference(utt, context, horizon)
        else:
            return self._single_horizon_inference(utt, context, horizon)

    def _single_horizon_inference(self, utt, context, horizon):
        """Bayesian inference for a single known horizon."""
        # compute P(utt | world, context, horizon) for each possible world
        likelihoods = [
            self.speaker.single_utterance_probability(
                utt, context, horizon=horizon, reward_weights=w
            )
            for w in self.possible_rewards
        ]

        # build DataFrame with world features and likelihoods
        df = pd.DataFrame(self.possible_rewards)
        df["likelihoods"] = likelihoods

        # normalize to get posterior (uniform prior cancels out)
        total = df["likelihoods"].sum()
        df["probability"] = df["likelihoods"] / total
        return df

    def _multihorizon_inference(self, utt, context, horizons):
        """Latent-horizon inference: marginalize over unknown horizon (Eq. 12).

        L1(w | s, u) ~ sum_H S1(u | w, s, H) * P(H) * P(w)
        """
        # collect posteriors for each horizon
        all_dfs = []
        for h in horizons:
            df_h = self._single_horizon_inference(utt, context, h)
            df_h["horizon"] = h
            all_dfs.append(df_h)

        # concatenate and re-normalize across all horizons
        combined = pd.concat(all_dfs, ignore_index=True)
        total = combined["likelihoods"].sum()
        combined["probability"] = combined["likelihoods"] / total
        return combined

    def point_estimate(self, posterior_df):
        """Compute expected value of each feature under the posterior.

        Returns a dict mapping each feature to its expected value.
        """
        result = {}
        for f in FEATURES:
            # E[w_f] = sum_worlds P(world) * world[f]
            result[f] = (posterior_df[f] * posterior_df["probability"]).sum()
        return result


print("Pragmatic Listener (L1) class defined.")

### 9.3 Stress Comparison Demo (cf. original paper Fig. 3)

The original paper's Fig. 3 compared L1's reward inferences for the same context under an instruction observation vs. a description observation — different utterance *types* led to different inferences.

Here we adapt the same idea for the stress extension: hold the lexical content fixed (`"take large red spotted"`) and vary only the **stress pattern** placed on the feature words. We compare four conditions: no stress, stress on size, stress on color, stress on texture.

The prediction: stressing a feature word should sharpen L1's marginal posterior over that feature's reward value, while leaving the other features relatively uncertain. This is the analog of the original paper's claim that different utterance types carry different information about reward — except now the "type" axis is prosodic rather than lexical.

We construct the context so each mushroom matches the instruction on exactly one feature dimension. That way each stress condition cleanly singles out a different mushroom for L0 to pick, and the resulting feature-count signal is informative about a different reward dimension.

In [ ]:
# --- Stress comparison demo ---
# Same lexical content ("take large red spotted") under four stress patterns.
# Context is designed so each mushroom matches the instruction on EXACTLY one
# feature dimension; this lets stress on each dimension cleanly single out a
# different mushroom for L0 to pick.

stress_demo_context = [
    {"size": "large",  "color": "blue",  "texture": "striped"},   # matches size only
    {"size": "small",  "color": "red",   "texture": "solid"},     # matches color only
    {"size": "medium", "color": "green", "texture": "spotted"},   # matches texture only
]

stress_conditions = {
    "no stress":      {"size": 0, "color": 0, "texture": 0},
    "stress SIZE":    {"size": 1, "color": 0, "texture": 0},
    "stress COLOR":   {"size": 0, "color": 1, "texture": 0},
    "stress TEXTURE": {"size": 0, "color": 0, "texture": 1},
}


def run_l1_under_stress(stress_dict):
    """Run L1 inference for 'take large red spotted' with the given stress pattern."""
    utt = {"type": "instruction",
           "size": "large", "color": "red", "texture": "spotted",
           "stress": stress_dict}

    # Pre-compute speaker likelihoods OUTSIDE @infer (closure captures them).
    listen = get_listener(utt, stress_demo_context, horizon=1)

    @infer
    def _inf():
        return listen()

    return dict(_inf()), utt


print("Running L1 inference under each stress condition...")
stress_results = {label: run_l1_under_stress(stress)
                  for label, stress in stress_conditions.items()}

print(f"\nContext: {stress_demo_context}\n")
for label, (post, utt) in stress_results.items():
    print(f"=== {label}: {utt_to_string(utt)} ===")
    for f in VARYING:
        m = marginal(post, small_worlds, f)
        line = f"  {f:>7}:  " + "  ".join(f"{v:+d}: {m[v]:.3f}" for v in POSSIBLE_VALUES)
        print(line)
    print()

In [ ]:
# Side-by-side bar charts: stress condition (rows) x varying feature (cols)
fig, axes = plt.subplots(len(stress_conditions), len(VARYING),
                         figsize=(11, 9), sharey=True, sharex='col')

for i, (label, (post, utt)) in enumerate(stress_results.items()):
    for j, f in enumerate(VARYING):
        m = marginal(post, small_worlds, f)
        ax = axes[i, j]
        ax.bar([f"{v:+d}" for v in POSSIBLE_VALUES],
               [m[v] for v in POSSIBLE_VALUES],
               color='steelblue', edgecolor='black')
        if i == 0:
            ax.set_title(f'P({f}_value | u)', fontsize=11)
        if j == 0:
            ax.set_ylabel(label, fontsize=10)
        ax.set_ylim(0, 1.0)
        ax.grid(True, axis='y', alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel('reward value')

plt.suptitle(
    'L1 marginals under different stress patterns of "take large red spotted"\n'
    '(stressing a feature word should sharpen the posterior over that feature)',
    fontsize=12, y=1.0
)
plt.tight_layout()
plt.show()

## 10. Latent-Horizon Pragmatic Inference (Fig. 3)

A key insight from the original paper: the pragmatic listener can **jointly infer** the speaker's horizon and reward function (Eq. 12).

*The original demo compared horizon inferences for an instruction vs a description (instructions suggest short-horizon speakers, descriptions suggest long-horizon ones). Since this project uses instructions only, the description side is commented out.*

In [ ]:
# Flippy pragmatic listener that jointly infers reward AND horizon (Eq. 12).
def make_latent_horizon_listener_flippy(speaker, possible_rewards, horizons):
    """L1(w | s, u) ~ sum_H S1(u | w, s, H) * P(H) * P(w)

    Returns `get_listener(utt, context)` which precomputes the speaker
    log-likelihoods for every (world, horizon) pair OUTSIDE the @infer'd region
    and returns a no-arg closure to use inside @infer.
    """

    def get_listener(utt, context):
        # Pre-compute speaker log-probabilities for each (world, horizon) pair
        log_likelihoods = [
            [math.log(speaker.single_utterance_probability(
                utt, context, horizon=h, reward_weights=w
            ) + 1e-300)
             for w in possible_rewards]
            for h in horizons
        ]

        def listen():
            world_idx = draw_from(len(possible_rewards))
            h_idx = draw_from(len(horizons))
            factor(log_likelihoods[h_idx][world_idx])
            return (world_idx, horizons[h_idx])

        return listen

    return get_listener


# Demo with small worlds and small horizon set
demo_horizons = [1, 2, 3, 4, 5, 10]

get_latent_listener = make_latent_horizon_listener_flippy(
    S1_small, small_worlds, demo_horizons
)

# Instruction-only inference (description comparison commented out)
instruction_utt = {"type": "instruction",
                   "size": "small", "color": "red", "texture": "spotted",
                   "stress": {"size": 0, "color": 0, "texture": 0}}
# description_utt = {"type": "description", "feature": "spotted", "value": 1}

print("Computing latent-horizon inference...\n")

latent_listen_instr = get_latent_listener(instruction_utt, demo_state)


@infer
def infer_instruction():
    return latent_listen_instr()


# latent_listen_desc = get_latent_listener(description_utt, demo_state)
# @infer
# def infer_description():
#     return latent_listen_desc()


instr_posterior = dict(infer_instruction())
# desc_posterior = dict(infer_description())


def extract_horizon_marginal(posterior, horizons):
    """Marginalize over worlds to get P(H | utt)."""
    marginal = {h: 0.0 for h in horizons}
    for (world_idx, h), prob in posterior.items():
        marginal[h] += prob
    return marginal


instr_horizon = extract_horizon_marginal(instr_posterior, demo_horizons)
# desc_horizon  = extract_horizon_marginal(desc_posterior, demo_horizons)

print(f"Horizon posterior given '{utt_to_string(instruction_utt)}':")
for h in demo_horizons:
    bar = '#' * int(instr_horizon[h] * 40)
    print(f"  H={h:>2}: {instr_horizon[h]:.4f}  {bar}")

# print(f"\nHorizon posterior given '{utt_to_string(description_utt)}':")
# for h in demo_horizons:
#     bar = '#' * int(desc_horizon[h] * 40)
#     print(f"  H={h:>2}: {desc_horizon[h]:.4f}  {bar}")

In [ ]:
# Plot horizon posterior for the instruction (description side commented out)
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar([str(h) for h in demo_horizons], [instr_horizon[h] for h in demo_horizons],
       color='steelblue', edgecolor='black')
ax.set_title(f'Instruction: "{instruction_utt["size"]} {instruction_utt["color"]} {instruction_utt["texture"]}"',
             fontsize=11)
ax.set_xlabel('Speaker Horizon H')
ax.set_ylabel('P(H | utterance)')

plt.suptitle('Inferred Speaker Horizon (cf. Fig. 3)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# --- Original side-by-side instruction-vs-description plot (commented out) ---
# fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
# axes[0].bar([str(h) for h in demo_horizons], [instr_horizon[h] for h in demo_horizons],
#             color='steelblue', edgecolor='black')
# axes[0].set_title(f'Instruction: "{instruction_utt["color"]} {instruction_utt["texture"]}"', fontsize=11)
# axes[0].set_xlabel('Speaker Horizon H')
# axes[0].set_ylabel('P(H | utterance)')
# axes[1].bar([str(h) for h in demo_horizons], [desc_horizon[h] for h in demo_horizons],
#             color='coral', edgecolor='black')
# axes[1].set_title(f'Description: "{description_utt["feature"]} is {description_utt["value"]:+d}"', fontsize=11)
# axes[1].set_xlabel('Speaker Horizon H')
# plt.suptitle('Inferred Speaker Horizon (cf. Fig. 3)', fontsize=13, y=1.02)
# plt.tight_layout()
# plt.show()

## 11. Experiment Trial Generation

*Commented out for this project.* This section was infrastructure for running the model on the same 28 mushroom-patch trials that humans saw in the original paper's JavaScript experiment, including feature-display randomization for saliency control. We are **not doing model-vs-human comparison** in this project — there's no human data on prosodic stress — so this setup is unnecessary. The original trial JSON also has 2-feature mushrooms with capitalized display names, which is incompatible with our 3-feature schema.

In [ ]:
# --- Commented out: human-comparison trial setup (see section 11 markdown) ---
# import json
# import random
#
# # --- Feature Randomization ---
# # In the experiment, features were randomly mapped to display names
# # to avoid saliency biases (e.g., color more salient than texture)
#
# def randomize_features():
#     """Shuffle the mapping from canonical features to display features.
#
#     Returns a dict mapping canonical names to randomized display names.
#     This mirrors setup.js: features[0] and features[1] are shuffled color/texture arrays.
#     """
#     # shuffle within each feature type
#     shuffled_colors = random.sample(COLORS, len(COLORS))
#     shuffled_textures = random.sample(TEXTURES, len(TEXTURES))
#
#     # canonical order: green(+2), red(0), blue(-2) -> shuffled
#     color_map = dict(zip(COLORS, shuffled_colors))
#     # canonical order: spotted(+1), solid(0), striped(-1) -> shuffled
#     texture_map = dict(zip(TEXTURES, shuffled_textures))
#
#     mapping = {**color_map, **texture_map}
#     return mapping
#
#
# # Demo
# feature_map = randomize_features()
# print("Feature randomization example:")
# for canon, display in feature_map.items():
#     val = TRUE_REWARDS[canon]
#     print(f"  canonical '{canon}' (value {val:+d}) -> displayed as '{display}'")

In [ ]:
# --- Commented out: human-comparison trial setup (see section 11 markdown) ---
# # --- Load trial data from the original experiment ---
# # The JSON files contain 28 pre-generated trial states (patches of 3 mushrooms)
#
# trial_data_path = (
#     "how-to-talk/Javascript-Experiment/static/json/exp1_split1.json"
# )
#
# try:
#     with open(trial_data_path, 'r') as f:
#         trials_raw = json.load(f)
#     print(f"Loaded {len(trials_raw)} trials from {trial_data_path}")
#     print(f"Example trial: {json.dumps(trials_raw[0], indent=2)}")
# except FileNotFoundError:
#     print(f"Trial file not found at {trial_data_path}")
#     print("Generating synthetic trials instead...")
#     # generate 28 random states as fallback
#     trials_raw = [
#         {"action_context": [random.sample(ACTIONS, 3)], "trial_type": "select"}
#         for _ in range(28)
#     ]
#     print(f"Generated {len(trials_raw)} synthetic trials")

In [ ]:
# --- Commented out: human-comparison trial setup (see section 11 markdown) ---
# # --- Horizon Assignment ---
# # Each non-catch trial is randomly assigned a horizon (1, 2, or 4)
# # Balanced: 1/3 of trials at each horizon level
#
# def assign_horizons(n_trials, horizon_values=[1, 2, 4]):
#     """Create a balanced, shuffled list of horizon assignments.
#
#     Ensures equal representation of each horizon condition.
#     Mirrors the JavaScript: Array(n_per_h).fill(h) for each h, then shuffle.
#     """
#     n_per_horizon = n_trials // len(horizon_values)
#     # create balanced horizon list
#     horizons = []
#     for h in horizon_values:
#         horizons.extend([h] * n_per_horizon)
#     # fill any remainder with random horizons
#     while len(horizons) < n_trials:
#         horizons.append(random.choice(horizon_values))
#     # shuffle to randomize order
#     random.shuffle(horizons)
#     return horizons
#
#
# # Assign horizons to our trials
# trial_horizons = assign_horizons(len(trials_raw))
# print(f"Horizon distribution: {Counter(trial_horizons)}")

In [ ]:
# --- Commented out: human-comparison trial setup (see section 11 markdown) ---
# # --- Build complete trial list ---
# # Each trial combines a mushroom patch (state) with a horizon
#
# def build_trial(trial_data, horizon, feature_map, trial_idx):
#     """Build a single trial dict with all necessary information.
#
#     Includes both canonical and display (randomized) versions of the context.
#     """
#     # extract the mushroom patch (action context) from the trial data
#     canonical_context = trial_data["action_context"][0]
#
#     # apply feature randomization for display
#     display_context = [
#         {
#             "color": feature_map[m["color"]],
#             "texture": feature_map[m["texture"]]
#         }
#         for m in canonical_context
#     ]
#
#     # compute true values for each mushroom
#     values = [compute_reward(m, TRUE_REWARDS) for m in canonical_context]
#
#     return {
#         "trial_idx": trial_idx,
#         "trial_type": trial_data["trial_type"],
#         "horizon": horizon,
#         "canonical_context": canonical_context,     # for model computation
#         "display_context": display_context,          # what participants see
#         "mushroom_values": values,                   # true rewards
#         "best_action_idx": int(np.argmax(values)),   # optimal choice
#     }
#
#
# # Build all trials
# trials = [
#     build_trial(td, h, feature_map, i)
#     for i, (td, h) in enumerate(zip(trials_raw, trial_horizons))
# ]
#
# # Show a few trials
# print("Sample trials:")
# for t in trials[:3]:
#     print(f"\n  Trial {t['trial_idx']} (H={t['horizon']}):")
#     for j, m in enumerate(t['canonical_context']):
#         val = t['mushroom_values'][j]
#         best = ' <-- BEST' if j == t['best_action_idx'] else ''
#         print(f"    {m['color']:>5} {m['texture']:>7}  value={val:+d}{best}")

## 12. Simulating the Speaker's Choices

*Commented out for this project.* This section depends on the Section 11 trial list (human-comparison setup) and is not needed for our stress demo.

In [ ]:
# --- Commented out: depends on the Section 11 trial list (human-comparison setup) ---
# def simulate_speaker_trial(trial, speaker, top_k=3):
#     """Simulate the speaker's utterance choice for a single trial.
#
#     Returns the top-k most likely utterances with their probabilities.
#     """
#     context = trial["canonical_context"]
#     horizon = trial["horizon"]
#
#     # get speaker's distribution over all utterances
#     probs = speaker.utterance_probabilities(context, horizon=horizon)
#
#     # pair each utterance with its probability and sort
#     utt_probs = list(zip(speaker.utterances, probs))
#     utt_probs.sort(key=lambda x: -x[1])
#
#     # return top-k
#     return utt_probs[:top_k]
#
#
# # Create a speaker using the full utterance set
# L0_sim = LiteralListener(alpha_L=3)
# S1_sim = Speaker(L0_sim, alpha_S=10, utterances=ALL_UTTERANCES)
#
# # Simulate first 5 trials
# print("Speaker simulation for first 5 trials:")
# for trial in trials[:5]:
#     top_utts = simulate_speaker_trial(trial, S1_sim, top_k=3)
#     print(f"\n  Trial {trial['trial_idx']} (H={trial['horizon']}):")
#     print(f"    Context: {[(m['color'], m['texture']) for m in trial['canonical_context']]}")
#     for utt, prob in top_utts:
#         print(f"    {utt_to_string(utt):>45}  P={prob:.4f}")

## 13. Instruction vs Description Preference Across Horizons

*The original section replicated the paper's Fig. 2A: speakers shift from instructions to descriptions as horizon increases. Since this project uses instructions only, the comparison code below is commented out — to be replaced later with a stress vs no-stress preference plot.*

In [ ]:
# --- Original instruction-vs-description preference computation (commented out) ---
# def instruction_vs_description_ratio(speaker, context, horizon):
#     """Compute fraction of probability mass on instructions vs descriptions."""
#     probs = speaker.utterance_probabilities(context, horizon=horizon)
#     utts = speaker.utterances
#     instr_mass = sum(p for u, p in zip(utts, probs) if u["type"] == "instruction")
#     desc_mass  = sum(p for u, p in zip(utts, probs) if u["type"] == "description")
#     return instr_mass, desc_mass
#
#
# sample_states = ALL_STATES[:20]
# horizons = list(range(1, 11))
# avg_instr_frac = []
# avg_desc_frac  = []
#
# print("Computing instruction/description preference across horizons...")
# for h in horizons:
#     instr_fracs = []
#     desc_fracs  = []
#     for state in sample_states:
#         i_mass, d_mass = instruction_vs_description_ratio(S1_sim, state, h)
#         instr_fracs.append(i_mass)
#         desc_fracs.append(d_mass)
#     avg_instr_frac.append(np.mean(instr_fracs))
#     avg_desc_frac.append(np.mean(desc_fracs))
#     print(f"  H={h:>2}: instructions={avg_instr_frac[-1]:.3f}, descriptions={avg_desc_frac[-1]:.3f}")
# print("Done!")

In [ ]:
# --- Original instruction-vs-description plot (commented out) ---
# plt.figure(figsize=(8, 5))
# plt.plot(horizons, avg_instr_frac, 'k-', label='Instructions', linewidth=2)
# plt.plot(horizons, avg_desc_frac, 'k--', label='Descriptions', linewidth=2)
# plt.xlabel('Speaker Horizon H', fontsize=13)
# plt.ylabel('Utterance Probability', fontsize=13)
# plt.title('Speaker Preference: Instructions vs Descriptions (cf. Fig. S1)', fontsize=14)
# plt.legend(fontsize=12)
# plt.xticks(horizons)
# plt.ylim(0, 1.05)
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

## 14. Reinforcement Learning with Social Information

*Commented out for this project.* The original paper's Section 5.3 / Fig. 5 showed that a description-derived **prior** ("spotted is +1") accelerates Thompson-sampling RL. That demonstration depends on having a description channel to derive a prior from — we dropped descriptions, and there isn't a clean analogue with stressed instructions (stress amplifies inference *magnitude* rather than directly stating a feature value, so it doesn't translate into a Gaussian-prior mean shift). The implementation also has a 2-feature feature vector left over from the original (`size` would always be treated as 0), which would silently mis-train the learner if it were run.

If we wanted an RL-with-stress story we'd need to redesign — e.g., set the prior covariance to be tighter along the stressed feature axis. Out of scope for this project.

In [ ]:
# --- Commented out: RL with description-derived prior (see section 14 markdown) ---
# from scipy.stats import multivariate_normal
#
#
# class BayesianLearner:
#     """Thompson sampling learner with optional social prior from language.
#
#     Maintains a Gaussian belief over feature weights and updates it
#     via conjugate Bayesian updates after each observation.
#     """
#
#     def __init__(self, features, prior_mean=None, prior_var=5.0, noise_var=1.0):
#         self.features = features  # list of feature names
#         self.n = len(features)
#         self.noise_var = noise_var  # observation noise variance
#
#         # initialize Gaussian prior: N(prior_mean, prior_var * I)
#         if prior_mean is None:
#             self.mean = np.zeros(self.n)  # uninformative prior centered at 0
#         else:
#             self.mean = np.array(prior_mean)
#         self.cov = np.eye(self.n) * prior_var  # diagonal covariance
#
#     def sample_beliefs(self):
#         """Draw a sample from the current belief (for Thompson sampling)."""
#         sample = np.random.multivariate_normal(self.mean, self.cov)
#         # convert to feature dict
#         return {f: sample[i] for i, f in enumerate(self.features)}
#
#     def update(self, action, observed_reward):
#         """Bayesian update after observing reward from taking an action.
#
#         Uses conjugate Gaussian update:
#             posterior_precision = prior_precision + X X^T / noise_var
#             posterior_mean = posterior_cov * (prior_precision * prior_mean + X * y / noise_var)
#         """
#         # construct feature vector for the observed action
#         # NOTE: original implementation only references color/texture; size missing.
#         x = np.array([1.0 if f in [action["color"], action["texture"]] else 0.0
#                       for f in self.features])
#
#         # prior precision (inverse covariance)
#         prior_precision = np.linalg.inv(self.cov)
#
#         # posterior precision = prior precision + x x^T / noise_var
#         post_precision = prior_precision + np.outer(x, x) / self.noise_var
#
#         # posterior covariance
#         post_cov = np.linalg.inv(post_precision)
#
#         # posterior mean
#         post_mean = post_cov @ (prior_precision @ self.mean + x * observed_reward / self.noise_var)
#
#         self.mean = post_mean
#         self.cov = post_cov
#
#     def choose_action(self, context, beliefs=None):
#         """Choose the best action given current beliefs (greedy)."""
#         if beliefs is None:
#             beliefs = {f: self.mean[i] for i, f in enumerate(self.features)}
#         # pick the action with highest expected reward
#         rewards = [compute_reward(a, beliefs) for a in context]
#         return context[np.argmax(rewards)]
#
#
# print("BayesianLearner class defined.")

In [ ]:
# --- Commented out: RL with description-derived prior (see section 14 markdown) ---
# def run_thompson_sampling(learner, n_trials=25, true_rewards=None):
#     """Run Thompson sampling for n_trials, tracking regret at each step.
#
#     At each trial:
#     1. Sample a random state (mushroom patch)
#     2. Sample beliefs from posterior (Thompson sampling)
#     3. Choose the greedy action under sampled beliefs
#     4. Observe noisy reward
#     5. Update beliefs
#     6. Record regret (optimal - actual reward)
#     """
#     if true_rewards is None:
#         true_rewards = TRUE_REWARDS
#
#     regrets = []
#     cumulative_regret = 0
#
#     for t in range(n_trials):
#         # sample a random state
#         state = random.choice(ALL_STATES)
#
#         # Thompson sampling: draw beliefs from posterior
#         sampled_beliefs = learner.sample_beliefs()
#
#         # choose best action under sampled beliefs
#         chosen = learner.choose_action(state, sampled_beliefs)
#
#         # compute true reward + noise
#         true_r = compute_reward(chosen, true_rewards)
#         observed_r = true_r + np.random.normal(0, np.sqrt(learner.noise_var))
#
#         # compute regret = optimal reward - actual reward
#         optimal_r = max(compute_reward(a, true_rewards) for a in state)
#         regret = optimal_r - true_r
#         cumulative_regret += regret
#         regrets.append(cumulative_regret)
#
#         # update beliefs
#         learner.update(chosen, observed_r)
#
#     return regrets
#
#
# print("Thompson sampling runner defined.")

In [ ]:
# --- Commented out: RL with description-derived prior (see section 14 markdown) ---
# # Run comparison: Individual learner vs learner with social prior
# n_runs = 50       # number of independent runs to average
# n_trials = 25     # trials per run
#
# # --- Individual learner (no language) ---
# individual_regrets = []
# for _ in range(n_runs):
#     learner = BayesianLearner(FEATURES, prior_var=5.0, noise_var=1.0)
#     regrets = run_thompson_sampling(learner, n_trials)
#     individual_regrets.append(regrets)
#
# # --- Learner with description prior ---
# # Simulate receiving "spotted is +1" -> prior mean shifted for spotted feature
# desc_regrets = []
# for _ in range(n_runs):
#     # set prior mean to reflect the description
#     desc_prior = [0.0] * len(FEATURES)
#     desc_prior[FEATURES.index("spotted")] = 1.0  # spotted = +1 from description
#     learner = BayesianLearner(FEATURES, prior_mean=desc_prior, prior_var=3.0, noise_var=1.0)
#     regrets = run_thompson_sampling(learner, n_trials)
#     desc_regrets.append(regrets)
#
# # Average regret across runs
# avg_individual = np.mean(individual_regrets, axis=0)
# avg_desc = np.mean(desc_regrets, axis=0)
#
# print(f"Final avg cumulative regret:")
# print(f"  Individual (no language): {avg_individual[-1]:.2f}")
# print(f"  With description prior:   {avg_desc[-1]:.2f}")

In [ ]:
# --- Commented out: RL with description-derived prior (see section 14 markdown) ---
# # Plot regret curves (cf. Fig. 5)
# plt.figure(figsize=(8, 5))
# trials_axis = list(range(1, n_trials + 1))
#
# plt.plot(trials_axis, avg_individual, 'k-', label='Individual (no language)',
#          linewidth=2)
# plt.plot(trials_axis, avg_desc, 'b--', label='With description prior',
#          linewidth=2)
#
# plt.xlabel('Learning Trial', fontsize=13)
# plt.ylabel('Cumulative Regret', fontsize=13)
# plt.title('Regret with and without Social Information (cf. Fig. 5)', fontsize=14)
# plt.legend(fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

## 15. Flippy: Full RSA Pipeline Demo

*Commented out for this project.* This was a self-contained 4-world toy demonstration of generic RSA mechanics (description-based speaker + pragmatic listener), separate from the main pipeline. It's pedagogical filler that uses descriptions, which we've dropped, so it contradicts the rest of the notebook. Our actual stress-aware pipeline is demonstrated in Section 9.3.

In [ ]:
# --- Commented out: self-contained mini RSA pipeline demo (uses descriptions) ---
# # --- Full RSA pipeline in flippy ---
# # We use a minimal world for tractability:
# # - 2 features (color, texture)
# # - 2 values each (-1, +1)
# # - 4 possible worlds
#
# # Minimal feature set
# mini_features = ["color_val", "texture_val"]
# mini_values = [-1, 1]
#
# # 4 possible worlds
# mini_worlds = [
#     {"color_val": cv, "texture_val": tv}
#     for cv, tv in itertools.product(mini_values, mini_values)
# ]
#
# # 4 possible descriptions
# mini_descriptions = [
#     ("color_val", v) for v in mini_values
# ] + [
#     ("texture_val", v) for v in mini_values
# ]
#
# print("Mini worlds:", mini_worlds)
# print("Mini descriptions:", mini_descriptions)

In [ ]:
# --- Commented out: self-contained mini RSA pipeline demo (uses descriptions) ---
# # RSA speaker: chooses a description to maximize informativeness
# alpha_speaker = 5.0
#
#
# @infer
# def rsa_speaker():
#     """Speaker who knows the true world and chooses an informative description."""
#     # true world: color_val = +1, texture_val = -1
#     true_world = {"color_val": 1, "texture_val": -1}
#
#     # uniformly sample a description to produce
#     desc_idx = draw_from(len(mini_descriptions))
#     feature, value = mini_descriptions[desc_idx]
#
#     # condition: descriptions should be truthful
#     # (the stated value should match the true value)
#     is_truthful = (true_world[feature] == value)
#     # strongly favor truthful utterances
#     factor(alpha_speaker if is_truthful else -alpha_speaker)
#
#     return desc_idx
#
#
# speaker_dist = dict(rsa_speaker())
# print("RSA Speaker distribution (true world: color=+1, texture=-1):")
# for idx, prob in sorted(speaker_dist.items(), key=lambda x: -x[1]):
#     feature, value = mini_descriptions[idx]
#     print(f"  '{feature} is {value:+d}': P = {prob:.4f}")

In [ ]:
# --- Commented out: self-contained mini RSA pipeline demo (uses descriptions) ---
# # RSA pragmatic listener: given a description, infer the world
# alpha_listener = 5.0
#
#
# @infer
# def rsa_pragmatic_listener():
#     """Pragmatic listener: observes 'color_val is +1' and infers the world.
#
#     Reasons: a rational speaker would say this if color_val = +1 in the true world.
#     The listener should infer that color_val = +1, and remain uncertain about texture_val.
#     """
#     # prior: sample a possible world uniformly
#     world_idx = draw_from(len(mini_worlds))
#     world = mini_worlds[world_idx]
#
#     # observed description: "color_val is +1"
#     observed_feature = "color_val"
#     observed_value = 1
#
#     # for each possible description the speaker could have said,
#     # compute how likely the speaker would say the observed one
#     # in this hypothetical world
#     for feat, val in mini_descriptions:
#         is_truthful = (world[feat] == val)
#         score = alpha_listener if is_truthful else -alpha_listener
#         # the observed description is (color_val, +1)
#         if feat == observed_feature and val == observed_value:
#             # factor in the speaker likelihood for the observed utterance
#             factor(score)
#
#     return world_idx
#
#
# listener_dist = dict(rsa_pragmatic_listener())
# print("Pragmatic Listener posterior given 'color_val is +1':")
# for idx in sorted(listener_dist.keys()):
#     world = mini_worlds[idx]
#     prob = listener_dist[idx]
#     bar = '#' * int(prob * 40)
#     print(f"  {world}  P = {prob:.4f}  {bar}")

## 16. Summary

This notebook translated the JavaScript experiment and Python models from *"How to Talk So AI Will Learn"* into a unified implementation using **flippy** for probabilistic inference. Key components:

| Component | Original (JS/Python) | Flippy Version |
|-----------|---------------------|----------------|
| **Literal Listener (L0)** | Class with softmax | Class with softmax (same, deterministic) |
| **Speaker (S1)** | Softmax over utilities | `@infer` + `draw_from` + `factor(utility)` |
| **Pragmatic Listener (L1)** | Explicit Bayes' rule | `@infer` + `draw_from(worlds)` + `factor(log likelihood)` |
| **Latent Horizon** | Loop + marginalize | `draw_from(horizons)` as another random variable |
| **Thompson Sampling** | Gaussian updates | `BayesianLearner` class with conjugate updates |

Flippy's advantage: the RSA recursive reasoning structure maps naturally onto probabilistic programs — sampling from priors and conditioning on observations — rather than requiring manual Bayesian computations.